In [1]:
# =============================================================================
# Playground Series S6E5 — Predicting F1 Pit Stops
# FULL GPU MULTI : CatBoost (GPU 0) + XGBoost (GPU 1) en parallèle
# Optuna parallel tuning -> 5-fold parallel -> Blend pondéré
# WITH: Standard logging, error handling, intermediate persistence, VRAM monitor
# =============================================================================

import gc
import logging
import os
import json
import traceback
import time
import warnings
from datetime import datetime
from threading import Thread
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")

IS_KAGGLE = os.path.exists("/kaggle")
DATA_DIR = (
    "/kaggle/input/competitions/playground-series-s6e5" if IS_KAGGLE else "data/raw"
)
PATHS = {
    "train": f"{DATA_DIR}/train.csv",
    "test": f"{DATA_DIR}/test.csv",
    "original": (
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
        if IS_KAGGLE
        else "data/raw/f1_strategy_dataset_v4.csv"
    ),
    "sample_sub": f"{DATA_DIR}/sample_submission.csv",
}
OUT_DIR = "/kaggle/working" if IS_KAGGLE else "output"

TARGET = "PitNextLap"
ID_COL = "id"
SEED = 42
N_SPLITS = 5
USE_ORIGINAL = True
OPTUNA_TRIALS = 15
OPTUNA_TIMEOUT = 2700
OPTUNA_SAMPLE_FRAC = 0.15

N_GPUS = 0
if IS_KAGGLE:
    try:
        import subprocess

        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
        )
        N_GPUS = len(out.strip().split("\n"))
    except Exception:
        N_GPUS = 1
else:
    N_GPUS = 0

CAT_GPU = "0" if N_GPUS >= 1 else None
XGB_DEVICE = "cuda:0" if N_GPUS == 1 else ("cuda:1" if N_GPUS >= 2 else "cpu")
PARALLEL_TRAIN = N_GPUS >= 2


# ─── LOGGING SETUP ───────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

logger = logging.getLogger("pipeline")
logger.setLevel(logging.INFO)
logger.propagate = False

_fmt = logging.Formatter(
    "%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
)

_fh = logging.FileHandler(f"{OUT_DIR}/training_log.txt", mode="w")
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

_sh = logging.StreamHandler()
_sh.setFormatter(_fmt)
logger.addHandler(_sh)


def log_vram(tag: str = ""):
    if not IS_KAGGLE or N_GPUS == 0:
        return
    try:
        import subprocess as sp

        out = sp.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,memory.used,memory.total,memory_free",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        )
        lines = out.strip().split("\n")
        for line in lines:
            parts = [p.strip() for p in line.split(",")]
            if len(parts) == 4:
                logger.info(
                    f"VRAM {tag} | GPU {parts[0]}: "
                    f"{parts[1]}MB / {parts[2]}MB (free {parts[3]}MB)"
                )
    except Exception:
        pass


def fixed_bin(series, bins, name):
    vals = pd.cut(series, bins=bins, labels=False, include_lowest=True)
    return vals.fillna(-1).astype(int).astype(str).radd(f"{name}_")


# ─── CHARGEMENT ──────────────────────────────────────────────────────────────


def load_data():
    logger.info("=" * 60)
    logger.info("CHARGEMENT DES DONNEES")
    logger.info("=" * 60)
    try:
        train = pd.read_csv(PATHS["train"])
        test = pd.read_csv(PATHS["test"])
        sample = pd.read_csv(PATHS["sample_sub"])
    except FileNotFoundError as e:
        logger.error(f"Fichier introuvable : {e}")
        raise SystemExit(1)
    except Exception as e:
        logger.error(f"Erreur lors du chargement des donnees : {e}")
        raise

    original = None
    if USE_ORIGINAL and os.path.exists(PATHS["original"]):
        try:
            original = pd.read_csv(PATHS["original"])
            logger.info(f"Original dataset charge : {original.shape}")
        except Exception as e:
            logger.warning(f"Impossible de charger l'original : {e}")
    else:
        logger.info("Dataset original non utilise ou introuvable")

    logger.info(f"Train : {train.shape} | Test : {test.shape}")
    logger.info(
        f"Target : {TARGET} | "
        f"positif={train[TARGET].mean():.3f} | "
        f"nulls={train.isnull().sum().sum()}"
    )
    return train, test, sample, original


# ─── EDA ─────────────────────────────────────────────────────────────────────


def run_eda(train):
    logger.info("EDA rapide")
    num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
    if ID_COL in num_cols:
        num_cols.remove(ID_COL)
    logger.info(f"Numeriques : {len(num_cols)} | Categorielles : {len(cat_cols)}")

    try:
        if "Compound" in train.columns:
            logger.info(
                f"Pit par Compound : "
                f"{train.groupby('Compound')[TARGET].mean().round(3).to_dict()}"
            )
    except Exception as e:
        logger.warning(f"EDA partielle : {e}")


# ─── FEATURE ENGINEERING ────────────────────────────────────────────────────


def build_features(df):
    df = df.copy()
    if "LapTime (s)" in df.columns:
        df["LapTime_s"] = df["LapTime (s)"]
    df["EstimatedLaps"] = df["LapNumber"] / df["RaceProgress"].clip(1e-3)
    df["TyreAgeRatio"] = df["TyreLife"] / df["LapNumber"].clip(1)
    df["DegPerLap"] = df["Cumulative_Degradation"] / df["LapNumber"].clip(1)
    df["DriverRace"] = df["Driver"].astype(str) + "_" + df["Race"].astype(str)
    df["DriverCompound"] = df["Driver"].astype(str) + "_" + df["Compound"].astype(str)
    df["RaceCompound"] = df["Race"].astype(str) + "_" + df["Compound"].astype(str)
    df["CompoundStint"] = df["Compound"].astype(str) + "_S" + df["Stint"].astype(str)
    df["LapsRemaining_est"] = df["EstimatedLaps"] - df["LapNumber"]
    df["TyreLife_frac_race"] = df["TyreLife"] / df["EstimatedLaps"].clip(1)
    df["TyreLife_to_remaining"] = df["TyreLife"] / (df["LapsRemaining_est"] + 1).clip(1)
    df["Lap_to_total_laps"] = df["LapNumber"] / df["EstimatedLaps"].clip(1)
    df["Stint_to_lap"] = df["Stint"] / df["LapNumber"].clip(1)
    df["Abs_Position_Change"] = df["Position_Change"].abs()
    df["LapTime_Delta_abs"] = df["LapTime_Delta"].abs()
    df["LapTime_Delta_sign"] = np.sign(df["LapTime_Delta"])
    df["Cumulative_Degradation_abs"] = df["Cumulative_Degradation"].abs()
    df["Degradation_per_TyreLife"] = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)
    df["Delta_per_TyreLife"] = df["LapTime_Delta"] / (df["TyreLife"] + 1)
    df["Delta_per_Lap"] = df["LapTime_Delta"] / (df["LapNumber"] + 1)
    df["Position_x_RaceProgress"] = df["Position"] * df["RaceProgress"]
    df["Position_x_TyreLife"] = df["Position"] * df["TyreLife"]
    df["Stint_x_TyreLife"] = df["Stint"] * df["TyreLife"]
    df["PitStop_x_TyreLife"] = df["PitStop"] * df["TyreLife"]
    df["PitStop_x_Stint"] = df["PitStop"] * df["Stint"]
    df["LateRace_TyreLife"] = (df["RaceProgress"] > 0.65).astype(int) * df["TyreLife"]
    df["TyreLife_bin"] = fixed_bin(
        df["TyreLife"], [-1, 3, 6, 10, 15, 22, 30, 45, 80, 200], "tyre"
    )
    df["LapNumber_bin"] = fixed_bin(
        df["LapNumber"], [-1, 5, 12, 20, 30, 42, 55, 70, 120], "lap"
    )
    df["RaceProgress_bin"] = fixed_bin(
        df["RaceProgress"], [-0.01, 0.1, 0.25, 0.4, 0.55, 0.7, 0.85, 1.01], "progress"
    )
    df["Position_bin"] = fixed_bin(df["Position"], [0, 3, 6, 10, 14, 20, 99], "pos")
    return df


def engineer_features(train, test, original):
    logger.info("=" * 60)
    logger.info("FEATURE ENGINEERING")
    logger.info("=" * 60)
    train_fe = build_features(train)
    test_fe = build_features(test)
    drop_cols = [ID_COL, TARGET]
    feature_cols = [c for c in train_fe.columns if c not in drop_cols]
    cat_cols = train_fe[feature_cols].select_dtypes(include=["object"]).columns.tolist()
    train_fe[cat_cols] = train_fe[cat_cols].astype(str)
    test_fe[cat_cols] = test_fe[cat_cols].astype(str)
    logger.info(f"Features : {len(feature_cols)} | Categorielles : {len(cat_cols)}")
    logger.info(f"Train : {train_fe.shape} | Test : {test_fe.shape}")

    orig_df = None
    if original is not None:
        try:
            orig_df = original.copy()
            for col in ["Normalized_TyreLife"]:
                if col in orig_df.columns:
                    orig_df.drop(columns=[col], inplace=True, errors="ignore")
            if "LapTime (s)" in orig_df.columns:
                orig_df.rename(columns={"LapTime (s)": "LapTime_s"}, inplace=True)
            orig_df = build_features(orig_df)
            for c in feature_cols:
                if c not in orig_df.columns:
                    orig_df[c] = np.nan
            extra = [
                c
                for c in orig_df.columns
                if c not in feature_cols + [TARGET] and c != ID_COL
            ]
            if extra:
                orig_df.drop(columns=extra, inplace=True, errors="ignore")
            orig_df = orig_df[
                feature_cols + ([TARGET] if TARGET in orig_df.columns else [])
            ].copy()
            orig_df[cat_cols] = orig_df[cat_cols].astype(str)
            logger.info(f"Original aligned : {orig_df.shape}")
        except Exception as e:
            logger.warning(f"Erreur alignement original, on skip : {e}")
            orig_df = None

    return train_fe, test_fe, feature_cols, cat_cols, orig_df


# ─── CONSTRUCTEURS DE MODELES ────────────────────────────────────────────────


def make_catboost(params, seed, gpu_id=None):
    from catboost import CatBoostClassifier

    task_type = "GPU" if N_GPUS >= 1 else "CPU"
    cb_kwargs = dict(
        iterations=params.get("iterations", 5000),
        learning_rate=params["learning_rate"],
        depth=params["depth"],
        l2_leaf_reg=params["l2_leaf_reg"],
        random_strength=params["random_strength"],
        bagging_temperature=params["bagging_temperature"],
        border_count=params.get("border_count", 254),
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        od_type="Iter",
        od_wait=250,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
        task_type=task_type,
    )
    if task_type == "GPU":
        dev = gpu_id if gpu_id is not None else CAT_GPU
        cb_kwargs["devices"] = str(dev)
    return CatBoostClassifier(**cb_kwargs)


def make_xgboost(params, seed, gpu_id=None):
    from xgboost import XGBClassifier

    device = XGB_DEVICE
    if gpu_id is not None and N_GPUS >= 1:
        device = f"cuda:{gpu_id}"

    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=params.get("n_estimators", 5000),
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        tree_method="hist",
        device=device,
        enable_categorical=True,
        max_cat_to_onehot=16,
        random_state=seed,
        n_jobs=-1,
        early_stopping_rounds=250,
    )


# ─── HELPERS ─────────────────────────────────────────────────────────────────


def _prepare_fold(X, y, cat_cols, orig_df, tr_idx, va_idx):
    X_tr = X.iloc[tr_idx].reset_index(drop=True)
    y_tr = y.iloc[tr_idx].reset_index(drop=True)
    X_va = X.iloc[va_idx].reset_index(drop=True)
    y_va = y.iloc[va_idx].reset_index(drop=True)
    if orig_df is not None and TARGET in orig_df.columns:
        orig_y = orig_df[TARGET].copy()
        orig_x = orig_df.drop(columns=[TARGET]).copy()
        orig_x = orig_x.reindex(columns=X_tr.columns, fill_value=np.nan)
        X_tr = pd.concat([X_tr, orig_x], ignore_index=True)
        y_tr = pd.concat([y_tr, orig_y], ignore_index=True)
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols].astype(str)).astype(int)
    X_va[cat_cols] = enc.transform(X_va[cat_cols].astype(str)).astype(int)
    return X_tr, y_tr, X_va, y_va


def _to_cat_dtype(X, cat_cols):
    X_c = X.copy()
    for c in cat_cols:
        X_c[c] = X_c[c].astype("category")
    return X_c


# ─── OPTUNA ──────────────────────────────────────────────────────────────────


def optuna_objective(model_name, X, y, cat_cols, orig_df, gpu_id=None):
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    rng = np.random.RandomState(SEED)
    n_sample = int(len(X) * OPTUNA_SAMPLE_FRAC)
    sample_idx = rng.choice(len(X), size=n_sample, replace=False)
    X_s = X.iloc[sample_idx].reset_index(drop=True)
    y_s = y.iloc[sample_idx].reset_index(drop=True)

    def objective(trial):
        try:
            if model_name == "catboost":
                params = {
                    "iterations": 1000,
                    "learning_rate": trial.suggest_float(
                        "learning_rate", 0.01, 0.1, log=True
                    ),
                    "depth": trial.suggest_int("depth", 4, 10),
                    "l2_leaf_reg": trial.suggest_float(
                        "l2_leaf_reg", 0.1, 20.0, log=True
                    ),
                    "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
                    "bagging_temperature": trial.suggest_float(
                        "bagging_temperature", 0.0, 2.0
                    ),
                    "border_count": trial.suggest_int("border_count", 32, 255),
                }
            elif model_name == "xgboost":
                params = {
                    "n_estimators": 5000,
                    "learning_rate": trial.suggest_float(
                        "learning_rate", 0.005, 0.05, log=True
                    ),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float(
                        "colsample_bytree", 0.3, 1.0
                    ),
                    "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                    "reg_lambda": trial.suggest_float(
                        "reg_lambda", 1e-3, 10.0, log=True
                    ),
                }
            else:
                raise ValueError(model_name)

            folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
            oof = np.zeros(len(X_s), dtype=float)
            for tr_idx, va_idx in folds.split(X_s, y_s):
                X_tr, y_tr, X_va, y_va = _prepare_fold(
                    X_s, y_s, cat_cols, orig_df, tr_idx, va_idx
                )
                if model_name == "catboost":
                    model = make_catboost(params, SEED, gpu_id=gpu_id)
                    model.fit(
                        X_tr.copy(),
                        y_tr,
                        cat_features=cat_cols,
                        eval_set=(X_va.copy(), y_va),
                        use_best_model=True,
                        verbose=0,
                    )
                    oof[va_idx] = model.predict_proba(X_va)[:, 1]
                    del model
                elif model_name == "xgboost":
                    X_tr_c = _to_cat_dtype(X_tr, cat_cols)
                    X_va_c = _to_cat_dtype(X_va, cat_cols)
                    model = make_xgboost(params, SEED, gpu_id=gpu_id)
                    model.fit(X_tr_c, y_tr, eval_set=[(X_va_c, y_va)], verbose=0)
                    oof[va_idx] = model.predict_proba(X_va_c)[:, 1]
                    del model
                gc.collect()
            return roc_auc_score(y_s, oof)
        except Exception as e:
            logger.warning(f"Optuna trial echoue ({model_name}): {e}")
            return 0.0

    return objective


def run_optuna(model_name, X, y, cat_cols, orig_df, gpu_id=None):
    import optuna

    logger.info(f"OPTUNA — {model_name.upper()} ({OPTUNA_TRIALS} trials, GPU {gpu_id})")
    t0 = time.time()

    try:
        study = optuna.create_study(
            study_name=model_name,
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=SEED),
        )

        def _log_trial(study, trial):
            logger.info(
                f"[{model_name.upper()}] Trial {trial.number + 1}/{OPTUNA_TRIALS} "
                f"AUC={trial.value:.5f} | Best={study.best_value:.5f}"
            )

        study.optimize(
            optuna_objective(model_name, X, y, cat_cols, orig_df, gpu_id=gpu_id),
            n_trials=OPTUNA_TRIALS,
            timeout=OPTUNA_TIMEOUT,
            show_progress_bar=False,
            callbacks=[_log_trial],
        )
        elapsed = time.time() - t0
        logger.info(f"Optuna {model_name} termine en {elapsed:.0f}s")
        logger.info(f"Best {model_name} AUC : {study.best_value:.6f}")
        logger.info(f"Best params :\n{json.dumps(study.best_params, indent=2)}")
        return study.best_params
    except Exception as e:
        logger.error(f"Optuna crash pour {model_name} : {e}")
        logger.info("Utilisation des params par defaut")
        if model_name == "catboost":
            return {
                "learning_rate": 0.05,
                "depth": 6,
                "l2_leaf_reg": 3.0,
                "random_strength": 1.0,
                "bagging_temperature": 1.0,
                "border_count": 128,
            }
        elif model_name == "xgboost":
            return {
                "learning_rate": 0.02,
                "max_depth": 6,
                "min_child_weight": 10,
                "subsample": 0.8,
                "colsample_bytree": 0.7,
                "reg_alpha": 0.1,
                "reg_lambda": 1.0,
            }
        raise


def run_optuna_all(model_names, X, y, cat_cols, orig_df):
    logger.info("=" * 60)
    logger.info(f"OPTUNA — {len(model_names)} modeles x {OPTUNA_TRIALS} trials")
    logger.info(f"GPUs : {N_GPUS} | CatBoost=GPU {CAT_GPU}, XGBoost={XGB_DEVICE}")
    logger.info(f"Parallel training : {PARALLEL_TRAIN}")
    logger.info("=" * 60)

    log_vram("avant Optuna")

    if PARALLEL_TRAIN:
        gpu_map = {"catboost": 0, "xgboost": 1}
        results = {}

        def _run(name):
            results[name] = run_optuna(
                name, X, y, cat_cols, orig_df, gpu_id=gpu_map.get(name)
            )

        threads = [Thread(target=_run, args=(name,)) for name in model_names]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        log_vram("apres Optuna")
        return results
    else:
        results = {}
        for name in model_names:
            gpu_id = 0 if N_GPUS >= 1 else None
            results[name] = run_optuna(name, X, y, cat_cols, orig_df, gpu_id=gpu_id)
        log_vram("apres Optuna")
        return results


# ─── ENTRAINEMENT FINAL (PARALLEL GPU) ──────────────────────────────────────


def train_folds(best_params, X, y, X_test, cat_cols, orig_df):
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_cat = np.zeros(len(X), dtype=float)
    oof_xgb = np.zeros(len(X), dtype=float)
    test_cat = np.zeros(len(X_test), dtype=float)
    test_xgb = np.zeros(len(X_test), dtype=float)

    completed_folds_cat = []
    completed_folds_xgb = []

    for fold, (tr_idx, va_idx) in enumerate(folds.split(X, y), 1):
        logger.info(f"{'=' * 40} Fold {fold}/{N_SPLITS} {'=' * 40}")
        fold_t0 = time.time()
        log_vram(f"debut fold {fold}")

        X_tr = X.iloc[tr_idx].reset_index(drop=True)
        y_tr = y.iloc[tr_idx].reset_index(drop=True)
        X_va = X.iloc[va_idx].reset_index(drop=True)
        y_va = y.iloc[va_idx].reset_index(drop=True)
        fold_test = X_test.copy()

        if orig_df is not None and TARGET in orig_df.columns:
            orig_y = orig_df[TARGET].copy()
            orig_x = orig_df.drop(columns=[TARGET]).copy()
            orig_x = orig_x.reindex(columns=X_tr.columns, fill_value=np.nan)
            X_tr = pd.concat([X_tr, orig_x], ignore_index=True)
            y_tr = pd.concat([y_tr, orig_y], ignore_index=True)

        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols].astype(str)).astype(int)
        X_va[cat_cols] = enc.transform(X_va[cat_cols].astype(str)).astype(int)
        fold_test[cat_cols] = enc.transform(fold_test[cat_cols].astype(str)).astype(int)

        cat_oof = [None]
        cat_test_pred = [None]
        xgb_oof = [None]
        xgb_test_pred = [None]

        cat_error = [None]
        xgb_error = [None]

        def train_cat():
            try:
                logger.info(f"Fold {fold} — Debut CatBoost GPU 0")
                cat = make_catboost(best_params["catboost"], SEED + fold, gpu_id=0)
                cat.fit(
                    X_tr.copy(),
                    y_tr,
                    cat_features=cat_cols,
                    eval_set=(X_va.copy(), y_va),
                    use_best_model=True,
                )
                cat_oof[0] = cat.predict_proba(X_va)[:, 1]
                cat_test_pred[0] = cat.predict_proba(fold_test)[:, 1]
                auc = roc_auc_score(y_va, cat_oof[0])
                logger.info(f"Fold {fold} — CAT AUC = {auc:.6f}")

                model_path = f"{OUT_DIR}/catboost_fold{fold}.pkl"
                joblib.dump(cat, model_path)
                logger.info(f"Fold {fold} — CatBoost sauvegarde : {model_path}")

                del cat
                gc.collect()
                log_vram(f"apres CAT fold {fold}")
            except Exception as e:
                cat_error[0] = e
                logger.error(
                    f"Fold {fold} — CatBoost ERREUR : {e}\n{traceback.format_exc()}"
                )

        def train_xgb():
            try:
                logger.info(f"Fold {fold} — Debut XGBoost GPU 1")
                xgb = make_xgboost(best_params["xgboost"], SEED + fold, gpu_id=1)
                X_tr_c = _to_cat_dtype(X_tr, cat_cols)
                X_va_c = _to_cat_dtype(X_va, cat_cols)
                ft_c = _to_cat_dtype(fold_test, cat_cols)
                xgb.fit(X_tr_c, y_tr, eval_set=[(X_va_c, y_va)], verbose=250)
                xgb_oof[0] = xgb.predict_proba(X_va_c)[:, 1]
                xgb_test_pred[0] = xgb.predict_proba(ft_c)[:, 1]
                auc = roc_auc_score(y_va, xgb_oof[0])
                logger.info(f"Fold {fold} — XGB AUC = {auc:.6f}")

                model_path = f"{OUT_DIR}/xgboost_fold{fold}.pkl"
                joblib.dump(xgb, model_path)
                logger.info(f"Fold {fold} — XGBoost sauvegarde : {model_path}")

                del xgb
                gc.collect()
                log_vram(f"apres XGB fold {fold}")
            except Exception as e:
                xgb_error[0] = e
                logger.error(
                    f"Fold {fold} — XGBoost ERREUR : {e}\n{traceback.format_exc()}"
                )

        if PARALLEL_TRAIN:
            t_cat = Thread(target=train_cat)
            t_xgb = Thread(target=train_xgb)
            t_cat.start()
            t_xgb.start()
            t_cat.join()
            t_xgb.join()
        else:
            train_cat()
            train_xgb()

        if cat_oof[0] is not None:
            oof_cat[va_idx] = cat_oof[0]
            test_cat += cat_test_pred[0] / N_SPLITS
            completed_folds_cat.append(fold)
        else:
            logger.warning(f"Fold {fold} — CatBoost SKIP (erreur : {cat_error[0]})")

        if xgb_oof[0] is not None:
            oof_xgb[va_idx] = xgb_oof[0]
            test_xgb += xgb_test_pred[0] / N_SPLITS
            completed_folds_xgb.append(fold)
        else:
            logger.warning(f"Fold {fold} — XGBoost SKIP (erreur : {xgb_error[0]})")

        elapsed = time.time() - fold_t0
        logger.info(
            f"Fold {fold} termine en {elapsed:.1f}s | "
            f"CAT folds OK: {completed_folds_cat} | XGB folds OK: {completed_folds_xgb}"
        )

        gc.collect()

    has_cat = len(completed_folds_cat) > 0
    has_xgb = len(completed_folds_xgb) > 0

    if not has_cat and not has_xgb:
        logger.critical("AUCUN modele n'a reussi sur aucun fold !")
        raise RuntimeError("Tous les modeles ont echoue — aucun resultat")

    n_cat = len(completed_folds_cat) if has_cat else 1
    n_xgb = len(completed_folds_xgb) if has_xgb else 1

    if has_cat:
        test_cat = test_cat * N_SPLITS / n_cat
    if has_xgb:
        test_xgb = test_xgb * N_SPLITS / n_xgb

    auc_cat = roc_auc_score(y, oof_cat) if has_cat else 0.0
    auc_xgb = roc_auc_score(y, oof_xgb) if has_xgb else 0.0

    logger.info("=" * 60)
    logger.info("SCORES OOF FINAUX")
    logger.info(
        f"CatBoost : {auc_cat:.6f} ({len(completed_folds_cat)}/{N_SPLITS} folds)"
    )
    logger.info(
        f"XGBoost  : {auc_xgb:.6f} ({len(completed_folds_xgb)}/{N_SPLITS} folds)"
    )

    if has_cat and has_xgb:
        w_cat = auc_cat / (auc_cat + auc_xgb)
        w_xgb = auc_xgb / (auc_cat + auc_xgb)
        oof_blend = w_cat * oof_cat + w_xgb * oof_xgb
        blend_auc = roc_auc_score(y, oof_blend)
        logger.info(f"Blend ({w_cat:.3f}*cat + {w_xgb:.3f}*xgb) AUC : {blend_auc:.6f}")
        final_test = w_cat * test_cat + w_xgb * test_xgb
    elif has_cat:
        w_cat, w_xgb = 1.0, 0.0
        blend_auc = auc_cat
        final_test = test_cat
        logger.warning("Seul CatBoost a reussi — blend = CatBoost seul")
    elif has_xgb:
        w_cat, w_xgb = 0.0, 1.0
        blend_auc = auc_xgb
        final_test = test_xgb
        logger.warning("Seul XGBoost a reussi — blend = XGBoost seul")
    else:
        raise RuntimeError("Aucun modele")

    joblib.dump(oof_cat, f"{OUT_DIR}/oof_cat.pkl")
    joblib.dump(oof_xgb, f"{OUT_DIR}/oof_xgb.pkl")
    joblib.dump(final_test, f"{OUT_DIR}/final_test.pkl")
    logger.info(f"Predictions intermediaires sauvegardees dans {OUT_DIR}/")

    return (
        (w_cat, w_xgb),
        final_test,
        blend_auc,
        {"cat": auc_cat, "xgb": auc_xgb, "blend": blend_auc},
    )


# ─── MAIN ────────────────────────────────────────────────────────────────────


def main():
    start_time = time.time()
    logger.info(f"Pipeline demarre — {datetime.now().isoformat()}")
    logger.info(f"GPUs detectes : {N_GPUS}")
    logger.info(f"CatBoost -> GPU {CAT_GPU}")
    logger.info(f"XGBoost  -> {XGB_DEVICE}")
    logger.info(f"Parallel : {PARALLEL_TRAIN}")

    train_raw = test_raw = sample = original = None
    best_params = None
    final_pred = None
    blend_auc = 0.0
    scores = {}

    try:
        train_raw, test_raw, sample, original = load_data()
    except SystemExit:
        return
    except Exception as e:
        logger.critical(f"Echec chargement donnees : {e}\n{traceback.format_exc()}")
        return

    try:
        run_eda(train_raw)
    except Exception as e:
        logger.warning(f"EDA echouee (non bloquant) : {e}")

    try:
        train_fe, test_fe, feature_cols, cat_cols, orig_df = engineer_features(
            train_raw, test_raw, original
        )
    except Exception as e:
        logger.critical(f"Feature engineering echoue : {e}\n{traceback.format_exc()}")
        return

    y = train_fe[TARGET].astype(int)
    X = train_fe[feature_cols].copy()
    X_test = test_fe[feature_cols].copy()
    logger.info(f"Train : {X.shape} | Test : {X_test.shape}")

    try:
        best_params = run_optuna_all(["catboost", "xgboost"], X, y, cat_cols, orig_df)
    except Exception as e:
        logger.error(f"Optuna echoue : {e}\n{traceback.format_exc()}")
        logger.info("Utilisation des hyperparametres par defaut")
        best_params = {
            "catboost": {
                "learning_rate": 0.05,
                "depth": 6,
                "l2_leaf_reg": 3.0,
                "random_strength": 1.0,
                "bagging_temperature": 1.0,
                "border_count": 128,
            },
            "xgboost": {
                "learning_rate": 0.02,
                "max_depth": 6,
                "min_child_weight": 10,
                "subsample": 0.8,
                "colsample_bytree": 0.7,
                "reg_alpha": 0.1,
                "reg_lambda": 1.0,
            },
        }

    logger.info("MEILLEURS HYPERPARAMETRES")
    logger.info(json.dumps(best_params, indent=2, default=str))

    logger.info("=" * 60)
    logger.info("ENTRAINEMENT FINAL PARALLEL 2xGPU (5-fold + Blend)")
    logger.info("=" * 60)

    try:
        weights, final_pred, blend_auc, scores = train_folds(
            best_params, X, y, X_test, cat_cols, orig_df
        )
    except Exception as e:
        logger.error(f"Entrainement folds echoue : {e}\n{traceback.format_exc()}")

        cached_test = f"{OUT_DIR}/final_test.pkl"
        if os.path.exists(cached_test):
            logger.info("Restauration depuis le cache final_test.pkl")
            final_pred = joblib.load(cached_test)
            weights = (0.5, 0.5)
        else:
            logger.warning("Pas de cache — fallback : prediction uniforme")
            final_pred = np.full(len(X_test), 0.5)
            weights = (0.5, 0.5)

    try:
        os.makedirs(OUT_DIR, exist_ok=True)
        joblib.dump(weights, f"{OUT_DIR}/blend_weights.pkl")
        if best_params:
            joblib.dump(best_params, f"{OUT_DIR}/best_params.pkl")
        logger.info(f"Artefacts sauvegardes dans {OUT_DIR}/")
    except Exception as e:
        logger.warning(f"Sauvegarde artefacts echouee : {e}")

    try:
        sub = sample.copy()
        pred_col = [c for c in sub.columns if c != ID_COL][0]
        sub[pred_col] = final_pred
        sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
        logger.info(f"Submission : {OUT_DIR}/submission.csv | Shape : {sub.shape}")
        logger.info(
            f"Stats : mean={sub[pred_col].mean():.4f} | "
            f"min={sub[pred_col].min():.4f} | max={sub[pred_col].max():.4f}"
        )
    except Exception as e:
        logger.critical(
            f"Generation submission echouee : {e}\n{traceback.format_exc()}"
        )

    elapsed = time.time() - start_time
    summary = {
        "competition": "playground-series-s6e5",
        "train_rows": int(len(train_raw)) if train_raw is not None else 0,
        "test_rows": int(len(test_raw)) if test_raw is not None else 0,
        "features": int(X.shape[1]),
        "cat_features": int(len(cat_cols)) if "cat_cols" in dir() else 0,
        "n_splits": N_SPLITS,
        "seed": SEED,
        "n_gpus": N_GPUS,
        "parallel": PARALLEL_TRAIN,
        "models": ["CatBoost-GPU0", "XGBoost-GPU1"],
        "optuna_trials": OPTUNA_TRIALS,
        "best_params": best_params,
        "blend_weights": {"cat": weights[0], "xgb": weights[1]} if weights else {},
        "oof_aucs": {k: float(v) for k, v in scores.items()},
        "elapsed_seconds": round(elapsed, 1),
    }
    logger.info(f"Resume :\n{json.dumps(summary, indent=2, default=str)}")
    logger.info(f"Pipeline termine en {elapsed:.0f}s — {datetime.now().isoformat()}")


if __name__ == "__main__":
    main()


2026-05-23 09:05:19 [INFO] Pipeline demarre — 2026-05-23T09:05:19.531499
2026-05-23 09:05:19 [INFO] GPUs detectes : 1
2026-05-23 09:05:19 [INFO] CatBoost -> GPU 0
2026-05-23 09:05:19 [INFO] XGBoost  -> cuda:0
2026-05-23 09:05:19 [INFO] Parallel : False
2026-05-23 09:05:19 [INFO] ============================================================
2026-05-23 09:05:19 [INFO] CHARGEMENT DES DONNEES
2026-05-23 09:05:19 [INFO] ============================================================
2026-05-23 09:05:20 [INFO] Dataset original non utilise ou introuvable
2026-05-23 09:05:20 [INFO] Train : (439140, 16) | Test : (188165, 15)
2026-05-23 09:05:20 [INFO] Target : PitNextLap | positif=0.199 | nulls=0
2026-05-23 09:05:20 [INFO] EDA rapide
2026-05-23 09:05:20 [INFO] Numeriques : 12 | Categorielles : 3
2026-05-23 09:05:20 [INFO] Pit par Compound : {'HARD': 0.328, 'INTERMEDIATE': 0.152, 'MEDIUM': 0.101, 'SOFT': 0.193, 'WET': 0.025}
2026-05-23 09:05:20 [INFO] ================================================

[0]	validation_0-auc:0.93216
[250]	validation_0-auc:0.94428
[500]	validation_0-auc:0.94661
[750]	validation_0-auc:0.94802
[1000]	validation_0-auc:0.94871
[1250]	validation_0-auc:0.94929
[1500]	validation_0-auc:0.94981
[1750]	validation_0-auc:0.95022
[2000]	validation_0-auc:0.95055
[2250]	validation_0-auc:0.95084
[2500]	validation_0-auc:0.95103
[2750]	validation_0-auc:0.95116
[3000]	validation_0-auc:0.95125
[3250]	validation_0-auc:0.95133
[3500]	validation_0-auc:0.95139
[3750]	validation_0-auc:0.95144
[4000]	validation_0-auc:0.95147
[4250]	validation_0-auc:0.95148
[4500]	validation_0-auc:0.95150
[4750]	validation_0-auc:0.95149
[4816]	validation_0-auc:0.95150


2026-05-23 09:49:03 [INFO] Fold 1 — XGB AUC = 0.951501
2026-05-23 09:49:05 [INFO] Fold 1 — XGBoost sauvegarde : /kaggle/working/xgboost_fold1.pkl
2026-05-23 09:49:05 [WARNING] Fold 1 — CatBoost SKIP (erreur : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr')
2026-05-23 09:49:05 [INFO] Fold 1 termine en 690.8s | CAT folds OK: [] | XGB folds OK: [1]
2026-05-23 09:49:05 [INFO] ======================================== Fold 2/5 ========================================
2026-05-23 09:49:07 [INFO] Fold 2 — Debut CatBoost GPU 0
2026-05-23 09:49:07 [ERROR] Fold 2 — CatBoost ERREUR : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr'
Traceback (most recent call last):
  File "/tmp/ipykernel_16/828454852.py", line 583, in train_cat
    cat.fit(
  File "/usr/local/lib/python3.12/dist-packages/catboost/core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, N

[0]	validation_0-auc:0.92386
[250]	validation_0-auc:0.94273
[500]	validation_0-auc:0.94518
[750]	validation_0-auc:0.94641
[1000]	validation_0-auc:0.94702
[1250]	validation_0-auc:0.94752
[1500]	validation_0-auc:0.94801
[1750]	validation_0-auc:0.94830
[2000]	validation_0-auc:0.94856
[2250]	validation_0-auc:0.94876
[2500]	validation_0-auc:0.94894
[2750]	validation_0-auc:0.94908
[3000]	validation_0-auc:0.94917
[3250]	validation_0-auc:0.94924
[3500]	validation_0-auc:0.94927
[3750]	validation_0-auc:0.94932
[4000]	validation_0-auc:0.94938
[4250]	validation_0-auc:0.94940
[4401]	validation_0-auc:0.94939


2026-05-23 09:59:29 [INFO] Fold 2 — XGB AUC = 0.949402
2026-05-23 09:59:30 [INFO] Fold 2 — XGBoost sauvegarde : /kaggle/working/xgboost_fold2.pkl
2026-05-23 09:59:30 [WARNING] Fold 2 — CatBoost SKIP (erreur : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr')
2026-05-23 09:59:30 [INFO] Fold 2 termine en 624.9s | CAT folds OK: [] | XGB folds OK: [1, 2]
2026-05-23 09:59:30 [INFO] ======================================== Fold 3/5 ========================================
2026-05-23 09:59:32 [INFO] Fold 3 — Debut CatBoost GPU 0
2026-05-23 09:59:32 [ERROR] Fold 3 — CatBoost ERREUR : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr'
Traceback (most recent call last):
  File "/tmp/ipykernel_16/828454852.py", line 583, in train_cat
    cat.fit(
  File "/usr/local/lib/python3.12/dist-packages/catboost/core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None

...

2026-05-23 10:08:58 [INFO] Fold 3 — XGB AUC = 0.950253
2026-05-23 10:08:59 [INFO] Fold 3 — XGBoost sauvegarde : /kaggle/working/xgboost_fold3.pkl
2026-05-23 10:08:59 [WARNING] Fold 3 — CatBoost SKIP (erreur : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr')
2026-05-23 10:08:59 [INFO] Fold 3 termine en 569.1s | CAT folds OK: [] | XGB folds OK: [1, 2, 3]
2026-05-23 10:08:59 [INFO] ======================================== Fold 4/5 ========================================
2026-05-23 10:09:01 [INFO] Fold 4 — Debut CatBoost GPU 0
2026-05-23 10:09:02 [ERROR] Fold 4 — CatBoost ERREUR : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr'
Traceback (most recent call last):
  File "/tmp/ipykernel_16/828454852.py", line 583, in train_cat
    cat.fit(
  File "/usr/local/lib/python3.12/dist-packages/catboost/core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, N

[0]	validation_0-auc:0.92378
[250]	validation_0-auc:0.94265
[500]	validation_0-auc:0.94509
[750]	validation_0-auc:0.94649
[1000]	validation_0-auc:0.94719
[1250]	validation_0-auc:0.94782
[1500]	validation_0-auc:0.94825
[1750]	validation_0-auc:0.94863
[2000]	validation_0-auc:0.94885
[2250]	validation_0-auc:0.94909
[2500]	validation_0-auc:0.94925
[2750]	validation_0-auc:0.94940
[3000]	validation_0-auc:0.94950
[3250]	validation_0-auc:0.94956
[3500]	validation_0-auc:0.94961
[3750]	validation_0-auc:0.94966
[4000]	validation_0-auc:0.94967
[4250]	validation_0-auc:0.94970
[4500]	validation_0-auc:0.94972
[4750]	validation_0-auc:0.94972
[4999]	validation_0-auc:0.94972


2026-05-23 10:21:01 [INFO] Fold 4 — XGB AUC = 0.949727
2026-05-23 10:21:02 [INFO] Fold 4 — XGBoost sauvegarde : /kaggle/working/xgboost_fold4.pkl
2026-05-23 10:21:02 [WARNING] Fold 4 — CatBoost SKIP (erreur : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr')
2026-05-23 10:21:02 [INFO] Fold 4 termine en 722.9s | CAT folds OK: [] | XGB folds OK: [1, 2, 3, 4]
2026-05-23 10:21:02 [INFO] ======================================== Fold 5/5 ========================================
2026-05-23 10:21:04 [INFO] Fold 5 — Debut CatBoost GPU 0
2026-05-23 10:21:05 [ERROR] Fold 5 — CatBoost ERREUR : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr'
Traceback (most recent call last):
  File "/tmp/ipykernel_16/828454852.py", line 583, in train_cat
    cat.fit(
  File "/usr/local/lib/python3.12/dist-packages/catboost/core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight

[0]	validation_0-auc:0.92548
[250]	validation_0-auc:0.94389
[500]	validation_0-auc:0.94629
[750]	validation_0-auc:0.94773
[1000]	validation_0-auc:0.94846
[1250]	validation_0-auc:0.94899
[1500]	validation_0-auc:0.94950
[1750]	validation_0-auc:0.94981
[2000]	validation_0-auc:0.95010
[2250]	validation_0-auc:0.95036
[2500]	validation_0-auc:0.95053
[2750]	validation_0-auc:0.95067
[3000]	validation_0-auc:0.95074
[3250]	validation_0-auc:0.95078
[3500]	validation_0-auc:0.95083
[3750]	validation_0-auc:0.95087
[4000]	validation_0-auc:0.95088
[4055]	validation_0-auc:0.95088


2026-05-23 10:31:31 [INFO] Fold 5 — XGB AUC = 0.950884
2026-05-23 10:31:32 [INFO] Fold 5 — XGBoost sauvegarde : /kaggle/working/xgboost_fold5.pkl
2026-05-23 10:31:32 [WARNING] Fold 5 — CatBoost SKIP (erreur : catboost/cuda/cuda_lib/cuda_manager.cpp:201: Condition violated: `State == nullptr')
2026-05-23 10:31:32 [INFO] Fold 5 termine en 629.9s | CAT folds OK: [] | XGB folds OK: [1, 2, 3, 4, 5]
2026-05-23 10:31:32 [INFO] ============================================================
2026-05-23 10:31:32 [INFO] SCORES OOF FINAUX
2026-05-23 10:31:32 [INFO] CatBoost : 0.000000 (0/5 folds)
2026-05-23 10:31:32 [INFO] XGBoost  : 0.950342 (5/5 folds)
2026-05-23 10:31:32 [WARNING] Seul XGBoost a reussi — blend = XGBoost seul
2026-05-23 10:31:32 [INFO] Predictions intermediaires sauvegardees dans /kaggle/working/
2026-05-23 10:31:32 [INFO] Artefacts sauvegardes dans /kaggle/working/
2026-05-23 10:31:33 [INFO] Submission : /kaggle/working/submission.csv | Shape : (188165, 2)
2026-05-23 10:31:33 [INF